# Temporary S5 FFT Verification

Standalone checks for the parallel FFT implementation of the diagonal S5 used by POSSM Stage-2. Run this before launching a long Stage-2 job.

In [ ]:
from pathlib import Path
import json
import sys
import time
from statistics import median

import torch

def find_repo_root() -> Path:
    starts = [Path.cwd(), Path('/content/utah-ssl'), Path('/content/drive/MyDrive/utah-ssl')]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'analysis' / 'active' / 'ssl_experiments').exists():
                return candidate
    raise FileNotFoundError('Could not find the utah-ssl repo root. cd into the repo first.')

REPO_ROOT = find_repo_root()
EXPERIMENTS_DIR = REPO_ROOT / 'analysis' / 'active' / 'ssl_experiments'
S5_DIR = REPO_ROOT / 'analysis' / 'active' / 'transfer_benchmark' / 'ssl_autoresearch'
for path in (REPO_ROOT, EXPERIMENTS_DIR, S5_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from s5 import BidirectionalS5SequenceBackbone, DiagonalS5SSM, S5SequenceBackbone
from possm_ssl.model import POSSMEncoder, POSSMPhonemeModel, causal_conv_output_lengths

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(7)
print('repo:', REPO_ROOT)
print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu:', torch.cuda.get_device_name(DEVICE))

## 1. Direct SSM Equivalence

In [ ]:
def assert_close(actual, expected, *, name, atol=1e-4, rtol=1e-3):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
    print(f'passed: {name}')

def direct_s5_output_check(device=DEVICE):
    torch.manual_seed(0)
    recurrent = DiagonalS5SSM(d_model=5, d_state=3, implementation='recurrent').to(device)
    fft = DiagonalS5SSM(d_model=5, d_state=3, implementation='fft').to(device)
    fft.load_state_dict(recurrent.state_dict())
    recurrent.eval()
    fft.eval()

    x = torch.randn(3, 7, 5, device=device)
    lengths = torch.tensor([7, 4, 2], dtype=torch.long, device=device)
    x[1, 4:] = 3.0
    x[2, 2:] = -2.0

    recurrent_out = recurrent(x, lengths)
    fft_out = fft(x, lengths)
    assert_close(fft_out, recurrent_out, name='direct S5 output with padded nonzero inputs')
    assert torch.equal(fft_out[1, 4:], torch.zeros_like(fft_out[1, 4:]))
    assert torch.equal(fft_out[2, 2:], torch.zeros_like(fft_out[2, 2:]))
    return float((fft_out - recurrent_out).abs().max().detach().cpu())

def direct_s5_gradient_check(device=DEVICE):
    torch.manual_seed(1)
    recurrent = DiagonalS5SSM(d_model=4, d_state=3, implementation='recurrent').to(device)
    fft = DiagonalS5SSM(d_model=4, d_state=3, implementation='fft').to(device)
    fft.load_state_dict(recurrent.state_dict())
    recurrent.eval()
    fft.eval()

    x_base = torch.randn(2, 6, 4, device=device)
    target = torch.randn(2, 6, 4, device=device)
    lengths = torch.tensor([6, 3], dtype=torch.long, device=device)

    recurrent_x = x_base.detach().clone().requires_grad_(True)
    fft_x = x_base.detach().clone().requires_grad_(True)
    recurrent_loss = (recurrent(recurrent_x, lengths) * target).sum()
    fft_loss = (fft(fft_x, lengths) * target).sum()
    recurrent_loss.backward()
    fft_loss.backward()

    assert_close(fft_x.grad, recurrent_x.grad, name='direct S5 input gradients', atol=5e-4, rtol=2e-3)
    recurrent_params = dict(recurrent.named_parameters())
    fft_params = dict(fft.named_parameters())
    max_param_grad_diff = 0.0
    for name in ('lambda_real_log', 'lambda_imag', 'log_dt', 'B_re', 'B_im', 'C_re', 'C_im', 'D.weight'):
        diff = (fft_params[name].grad - recurrent_params[name].grad).abs().max()
        max_param_grad_diff = max(max_param_grad_diff, float(diff.detach().cpu()))
        assert_close(fft_params[name].grad, recurrent_params[name].grad, name=f'gradient {name}', atol=5e-4, rtol=2e-3)
    return float((fft_x.grad - recurrent_x.grad).abs().max().detach().cpu()), max_param_grad_diff

output_diff = direct_s5_output_check()
input_grad_diff, param_grad_diff = direct_s5_gradient_check()
print(json.dumps({'max_output_diff': output_diff, 'max_input_grad_diff': input_grad_diff, 'max_param_grad_diff': param_grad_diff}, indent=2))

## 2. Backbone Equivalence

In [ ]:
def check_causal_backbone(num_layers: int, device=DEVICE):
    torch.manual_seed(10 + num_layers)
    recurrent = S5SequenceBackbone(
        d_model=5,
        d_state=3,
        num_layers=num_layers,
        dropout=0.0,
        ffn_multiplier=1.0,
        implementation='recurrent',
    ).to(device)
    fft = S5SequenceBackbone(
        d_model=5,
        d_state=3,
        num_layers=num_layers,
        dropout=0.0,
        ffn_multiplier=1.0,
        implementation='fft',
    ).to(device)
    fft.load_state_dict(recurrent.state_dict())
    recurrent.eval()
    fft.eval()

    x = torch.randn(2, 6, 5, device=device)
    lengths = torch.tensor([6, 4], dtype=torch.long, device=device)
    x[1, 4:] = 7.0
    recurrent_out = recurrent(x, lengths)
    fft_out = fft(x, lengths)
    assert_close(fft_out, recurrent_out, name=f'causal S5 backbone layers={num_layers}')
    assert torch.equal(fft_out[1, 4:], torch.zeros_like(fft_out[1, 4:]))
    return float((fft_out - recurrent_out).abs().max().detach().cpu())

def check_bidirectional_backbone(device=DEVICE):
    torch.manual_seed(12)
    recurrent = BidirectionalS5SequenceBackbone(
        d_model=4,
        d_state=3,
        num_layers=1,
        dropout=0.0,
        ffn_multiplier=1.0,
        implementation='recurrent',
    ).to(device)
    fft = BidirectionalS5SequenceBackbone(
        d_model=4,
        d_state=3,
        num_layers=1,
        dropout=0.0,
        ffn_multiplier=1.0,
        implementation='fft',
    ).to(device)
    fft.load_state_dict(recurrent.state_dict())
    recurrent.eval()
    fft.eval()

    x = torch.randn(2, 6, 4, device=device)
    lengths = torch.tensor([6, 3], dtype=torch.long, device=device)
    x[1, 3:] = -5.0
    recurrent_out = recurrent(x, lengths)
    fft_out = fft(x, lengths)
    assert_close(fft_out, recurrent_out, name='bidirectional S5 backbone')
    assert torch.equal(fft_out[1, 3:], torch.zeros_like(fft_out[1, 3:]))
    return float((fft_out - recurrent_out).abs().max().detach().cpu())

backbone_diffs = {
    'causal_layers_1': check_causal_backbone(1),
    'causal_layers_2': check_causal_backbone(2),
    'bidirectional_layers_1': check_bidirectional_backbone(),
}
print(json.dumps(backbone_diffs, indent=2))

## 3. POSSM Stage-2 Model Smoke

In [ ]:
torch.manual_seed(20)
base_encoder = POSSMEncoder(
    input_dim=5,
    model_dim=4,
    latent_count=2,
    ffn_hidden_size=16,
    dropout=0.0,
    feature_mode='tx_sbp',
).to(DEVICE)
model = POSSMPhonemeModel(
    base_encoder=base_encoder,
    vocab_size=6,
    decoder_backbone_type='s5',
    s5_hidden_size=8,
    s5_state_size=4,
    s5_num_layers=1,
    s5_dropout=0.0,
    s5_direction='causal',
    s5_ffn_multiplier=1.0,
    s5_implementation='fft',
    conv_kernel_size=3,
    conv_stride=2,
    conv_dropout=0.0,
).to(DEVICE)
model.train()

x = torch.randn(2, 8, 5, device=DEVICE)
lengths = torch.tensor([8, 5], dtype=torch.long, device=DEVICE)
outputs = model(x, lengths, session_ids=['session_a', 'session_b'])
expected_token_lengths = causal_conv_output_lengths(lengths, stride=2)
assert tuple(outputs['decoder_hidden'].shape) == (2, 8, 8)
assert tuple(outputs['logits'].shape) == (2, 4, 6)
assert torch.equal(outputs['token_lengths'], expected_token_lengths)
assert torch.equal(outputs['decoder_hidden'][1, 5:], torch.zeros_like(outputs['decoder_hidden'][1, 5:]))

loss = outputs['logits'].square().mean()
loss.backward()
bad_grads = [name for name, param in model.named_parameters() if param.grad is not None and not torch.isfinite(param.grad).all()]
assert not bad_grads, bad_grads
print('passed: POSSM S5 FFT forward/backward smoke')
print(json.dumps({'logits_shape': list(outputs['logits'].shape), 'token_lengths': outputs['token_lengths'].detach().cpu().tolist(), 'loss': float(loss.detach().cpu())}, indent=2))

## 4. Timing Probe

Use smaller settings first. For the current notebook smoke, try `D_MODEL=384`, `D_STATE=64`, `NUM_LAYERS=2`. For the larger screen, try `D_MODEL=768`, `D_STATE=128`, `NUM_LAYERS=5`.

In [ ]:
BATCH_SIZE = 32
SEQ_LEN = 512
D_MODEL = 384
D_STATE = 64
NUM_LAYERS = 2
FFN_MULTIPLIER = 1.0
WARMUP = 3
REPEATS = 10

def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)

def train_step(model, x_base, lengths, target):
    model.zero_grad(set_to_none=True)
    x = x_base.detach().clone().requires_grad_(True)
    output = model(x, lengths)
    loss = (output * target).mean()
    loss.backward()
    return float(loss.detach().cpu()), output.detach(), x.grad.detach().clone()

def time_model(model, x_base, lengths, target):
    for _ in range(WARMUP):
        train_step(model, x_base, lengths, target)
    sync()
    times = []
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        train_step(model, x_base, lengths, target)
        sync()
        times.append(time.perf_counter() - t0)
    return {
        'median_step_s': float(median(times)),
        'min_step_s': float(min(times)),
        'max_step_s': float(max(times)),
        'peak_memory_mb': float(torch.cuda.max_memory_allocated(DEVICE) / 1024**2) if DEVICE.type == 'cuda' else None,
    }

torch.manual_seed(30)
recurrent = S5SequenceBackbone(
    d_model=D_MODEL,
    d_state=D_STATE,
    num_layers=NUM_LAYERS,
    dropout=0.0,
    ffn_multiplier=FFN_MULTIPLIER,
    implementation='recurrent',
).to(DEVICE).eval()
fft = S5SequenceBackbone(
    d_model=D_MODEL,
    d_state=D_STATE,
    num_layers=NUM_LAYERS,
    dropout=0.0,
    ffn_multiplier=FFN_MULTIPLIER,
    implementation='fft',
).to(DEVICE).eval()
fft.load_state_dict(recurrent.state_dict())

lengths = torch.full((BATCH_SIZE,), SEQ_LEN, dtype=torch.long, device=DEVICE)
if BATCH_SIZE > 1:
    lengths[-1] = max(1, SEQ_LEN // 2)
x_base = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=DEVICE)
target = torch.randn_like(x_base)

_, recurrent_output, recurrent_grad = train_step(recurrent, x_base, lengths, target)
_, fft_output, fft_grad = train_step(fft, x_base, lengths, target)
agreement = {
    'max_abs_output_diff': float((fft_output - recurrent_output).abs().max().detach().cpu()),
    'max_abs_input_grad_diff': float((fft_grad - recurrent_grad).abs().max().detach().cpu()),
}

recurrent_timing = time_model(recurrent, x_base, lengths, target)
fft_timing = time_model(fft, x_base, lengths, target)
result = {
    'device': str(DEVICE),
    'batch_size': BATCH_SIZE,
    'seq_len': SEQ_LEN,
    'd_model': D_MODEL,
    'd_state': D_STATE,
    'num_layers': NUM_LAYERS,
    **agreement,
    'recurrent': recurrent_timing,
    'fft': fft_timing,
    'fft_speedup_vs_recurrent': recurrent_timing['median_step_s'] / fft_timing['median_step_s'],
}
print(json.dumps(result, indent=2))